<a href="https://colab.research.google.com/github/yeshaa23/zoo-asia-graph-ETSproject/blob/main/notebooks/Data_Joining_ETS_Graf_FIX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving dbpedia.csv to dbpedia.csv
Saving wikidata.csv to wikidata.csv


In [2]:
import pandas as pd
import unicodedata
import re

In [3]:
import io

# Load data
df_wd = pd.read_csv(io.StringIO(uploaded['wikidata.csv'].decode('utf-8')))
df_db = pd.read_csv(io.StringIO(uploaded['dbpedia.csv'].decode('utf-8')))

print("Kolom Wikidata:", df_wd.columns.tolist())
print("Kolom DBpedia :", df_db.columns.tolist())

Kolom Wikidata: ['zoo', 'zooLabel', 'country', 'countryLabel', 'region', 'regionLabel', 'coord', 'inception', 'website']
Kolom DBpedia : ['zooLabel', 'wikidata', 'abstract', 'locationLabel', 'lat', 'long']


In [4]:
def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    text = re.sub(r"\s+", " ", text)
    return text

In [5]:
if "QID" in df_db.columns:
    df_db = df_db.rename(columns={"QID": "qid"})
df_wd["zooLabel"] = df_wd["zooLabel"].apply(normalize_text)
df_db["zooLabel"] = df_db["zooLabel"].apply(normalize_text)

In [6]:
# Extract qid dari kolom 'zoo'
df_wd["qid"] = df_wd["zoo"].apply(lambda x: str(x).split('/')[-1] if pd.notna(x) and 'http' in str(x) else x)

# Extract qid dari kolom 'wikidata'
df_db["qid"] = df_db["wikidata"].apply(lambda x: str(x).split('/')[-1] if pd.notna(x) and 'http' in str(x) else x)

df_wd["qid"] = df_wd["qid"].astype(str).str.strip()
df_db["qid"] = df_db["qid"].astype(str).str.strip()

In [7]:
# cleaning data
df_wd["qid"] = df_wd["qid"].replace(["nan", "None", ""], pd.NA)
df_db["qid"] = df_db["qid"].replace(["nan", "None", ""], pd.NA)

In [8]:
df_wd = df_wd.drop_duplicates(subset=["qid", "zooLabel"])
df_db = df_db.drop_duplicates(subset=["qid", "zooLabel"])

In [9]:
# gabung data
join_qid = pd.merge(
    df_wd,
    df_db[["qid", "zooLabel", "abstract", "locationLabel", "lat", "long"]],
    on="qid",
    how="left",
    suffixes=("_wd", "_db")
)
print("Hasil join by qid:", join_qid.shape)
print(join_qid.head())

Hasil join by qid: (258, 15)
                                        zoo                       zooLabel_wd  \
0   http://www.wikidata.org/entity/Q3162430  saigon zoo and botanical gardens   
1    http://www.wikidata.org/entity/Q993926                     asahiyama zoo   
2  http://www.wikidata.org/entity/Q19610585                        ankara zoo   
3   http://www.wikidata.org/entity/Q1267646                         dusit zoo   
4   http://www.wikidata.org/entity/Q2343219                  ramat gan safari   

                               country countryLabel  \
0  http://www.wikidata.org/entity/Q881      Vietnam   
1   http://www.wikidata.org/entity/Q17        Japan   
2   http://www.wikidata.org/entity/Q43       Turkey   
3  http://www.wikidata.org/entity/Q869     Thailand   
4  http://www.wikidata.org/entity/Q801       Israel   

                                     region       regionLabel  \
0      http://www.wikidata.org/entity/Q1854  Ho Chi Minh City   
1    http://www.wikidat

In [10]:
# fallback
unmatched = join_qid[join_qid["zooLabel_db"].isna()].copy()

fallback = pd.merge(
    unmatched,
    df_db[["zooLabel", "abstract", "locationLabel", "lat", "long"]],
    left_on="zooLabel_wd",
    right_on="zooLabel",
    how="left"
)

# rename kolom fallback
fallback = fallback.rename(columns={
    "zooLabel": "zooLabel_fb",
    "abstract_y": "abstract_fb",
    "locationLabel_y": "locationLabel_fb",
    "lat_y": "lat_fb",
    "long_y": "long_fb"
})

fallback = fallback[[
    "qid",
    "zooLabel_fb",
    "abstract_fb",
    "locationLabel_fb",
    "lat_fb",
    "long_fb"
]]

print("Hasil fallback by label:", fallback.shape)
print(fallback.head())

Hasil fallback by label: (135, 6)
         qid zooLabel_fb  abstract_fb locationLabel_fb  lat_fb  long_fb
0  Q19610585         NaN          NaN              NaN     NaN      NaN
1   Q4167605         NaN          NaN              NaN     NaN      NaN
2  Q11325868         NaN          NaN              NaN     NaN      NaN
3  Q22118342         NaN          NaN              NaN     NaN      NaN
4   Q4315033         NaN          NaN              NaN     NaN      NaN


In [11]:
hasil = join_qid.merge(fallback, on="qid", how="left")

# isi yang kosong dari fallback
hasil["zooLabel_db"] = hasil["zooLabel_db"].fillna(hasil["zooLabel_fb"])
hasil["abstract"] = hasil["abstract"].fillna(hasil["abstract_fb"])
hasil["locationLabel"] = hasil["locationLabel"].fillna(hasil["locationLabel_fb"])
hasil["lat"] = hasil["lat"].fillna(hasil["lat_fb"])
hasil["long"] = hasil["long"].fillna(hasil["long_fb"])

In [12]:
# zooLabel final: prioritas Wikidata
hasil["zooLabel_final"] = hasil["zooLabel_wd"]

kolom_final = [
    "qid",
    "zooLabel_final",
    "countryLabel",
    "inception",
    "regionLabel",
    "lat",
    "long"
]

df_final = hasil[kolom_final].copy()

# Rename biar rapi
df_final = df_final.rename(columns={
    "zooLabel_final": "zooLabel"
})

In [13]:
# hapus baris tanpa qid atau tanpa nama zoo
df_final = df_final.dropna(subset=["qid", "zooLabel"])

In [14]:
# hapus duplikat final
df_final = df_final.drop_duplicates()

In [15]:
# simpan hasil join final
df_final.to_csv("zoo_asia_final.csv", index=False)

print("\nJumlah data Wikidata :", len(df_wd))
print("Jumlah data DBpedia  :", len(df_db))
print("Jumlah hasil final   :", len(df_final))

print("\nContoh hasil final:")
print(df_final.head(10))


Jumlah data Wikidata : 257
Jumlah data DBpedia  : 967
Jumlah hasil final   : 257

Contoh hasil final:
          qid                          zooLabel countryLabel  \
0    Q3162430  saigon zoo and botanical gardens      Vietnam   
1     Q993926                     asahiyama zoo        Japan   
2   Q19610585                        ankara zoo       Turkey   
3    Q1267646                         dusit zoo     Thailand   
4    Q2343219                  ramat gan safari       Israel   
5    Q2709506                     singapore zoo    Singapore   
6    Q2896297                          negevzoo       Israel   
8    Q4167605                      tashkent zoo   Uzbekistan   
9   Q11325868              north safari sapporo        Japan   
10  Q22118342                         q22118342        Japan   

               inception        regionLabel locationLabel        lat  \
0   1865-01-01T00:00:00Z   Ho Chi Minh City           NaN  10.788055   
1   1967-07-01T00:00:00Z          Asahikawa     

In [16]:
from google.colab import files
files.download("zoo_asia_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>